# 🔬 Radyoloji LLM Karşılaştırma — Tam Deney Pipeline'ı

Bu notebook, **makale yazılacak seviyede** gerçek bir deney ortamı kurar:

1. **Gerçek veri** (HuggingFace'ten VQA-RAD)
2. **5 Generalist + 3 Domain-Adaptive + 3 Specialist model lane'i**
3. **Otomatik metrik hesaplama** (BLEU, ROUGE-L, VQA Accuracy, Halüsinasyon)
4. **Prior Bias Testi** (siyah görüntü deneyi)
5. **Prompt Ablasyonu** (baseline / detailed / chain-of-thought karşılaştırması)
6. **Sonuç tabloları ve JSON kaydı**

---

⚠️ **Runtime:** Üst menüden **Runtime → Change runtime type → T4 GPU** seçin.

⏱️ **Toplam süre:** ~2-4 saat (tüm lane'ler)

💡 **İpucu:** Her lane'i ayrı koşmak için Bölüm 3'teki `RUN_LANES` ayarını düzenleyin.

---
## 1️⃣ Kurulum ve Ortam Hazırlığı

In [1]:
# Projeyi klonla
!git clone https://github.com/Hanketsu3/LLMComparison.git 2>/dev/null || (cd LLMComparison && git pull)
%cd LLMComparison

/content/LLMComparison


In [ ]:

# 1. PyTorch 2.4.1 kur (Flash Attention 2 bu versiyonla hazır wheel var)
# %pip install -q torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 --index-url https://download.pytorch.org/whl/cu121

# 2. Temel bağımlılıklar
%pip install -r requirements.txt

# 3. Sürüm çakışmalarını önlemek için kritik pin'ler
# - transformers 4.57.0 -> huggingface-hub < 1.0 ister
# - cudf-cu12/pylibcudf-cu12 -> pyarrow < 20 ister
%pip install -q --upgrade --no-cache-dir "transformers==4.57.0" "huggingface-hub>=0.34.0,<1.0" "pyarrow>=14,<20"

# 4. Diğer bağımlılıklar (huggingface_hub burada özellikle YOK)
%pip install -U -q accelerate bitsandbytes datasets qwen-vl-utils evaluate rouge-score scikit-learn einops timm

print("✅ Kurulum tamamlandı! (uyumlu sürümler sabitlendi)")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 29.3 MB/s eta 0:00:00:00:0100:01
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5c575b8e98b02d88823115cee15af71f766d1b135823f1a57ede3432588c59fe
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 132.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 11.9 MB/s eta 0

In [3]:
# GPU kontrolü
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ GPU bulunamadı! Runtime → Change runtime type → T4 GPU seçin.")
    raise RuntimeError("GPU gerekli!")

In [ ]:
# Proje modüllerini import et
import sys, os, json, gc, time, logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from PIL import Image
from tqdm.notebook import tqdm

# Proje path'i ayarla
if os.path.exists("src"):
    sys.path.insert(0, os.getcwd())

from src.utils.model_registry import MODEL_REGISTRY, load_model, print_model_table
from src.utils.prompt_manager import PromptManager
from src.evaluation.nlp_metrics import BLEUEvaluator, ROUGEEvaluator
from src.evaluation.vqa_metrics import VQAAccuracyEvaluator
from src.evaluation.hallucination import FactCheXckerEvaluator

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Hızlı sürüm kontrolü
import transformers, huggingface_hub, pyarrow
print(f"transformers: {transformers.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"pyarrow: {pyarrow.__version__}")

print("✅ Tüm modüller yüklendi.")

---
## 2️⃣ Gerçek Veri İndirme (HuggingFace VQA-RAD)

**VQA-RAD**: 315 gerçek radyoloji görüntüsü, 3515 soru-cevap çifti.
Chest, abdomen, head bölgelerini kapsıyor.

In [5]:
from src.data.hf_vqa_rad import HFVQARADDataset

# Gerçek veri setini indir
NUM_SAMPLES = None  # Tüm test seti için None yapın

dataset = HFVQARADDataset(split="test", max_samples=NUM_SAMPLES)

stats = dataset.get_statistics()
print(f"\n📊 Veri Seti İstatistikleri:")
for k, v in stats.items():
    print(f"   {k}: {v}")

# İlk 3 örneği göster
print("\n" + "="*60)
for i in range(min(3, len(dataset))):
    s = dataset[i]
    print(f"\n📋 Örnek #{i+1}")
    print(f"   Soru: {s['question']}")
    print(f"   Cevap: {s['answer']}")
    display(s['image'])

---
## 3️⃣ Deney Konfigürasyonu

Hangi modelleri test edeceğimizi ve deney parametrelerini burada belirliyoruz.

---
## 🔑 HuggingFace Erişimi
`llama3-vision` artık aktif lane'de kullanılabilir. Gated Meta repo erişimin varsa aşağıdaki login hücresini çalıştır.

In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ============================================================
# DENEY KONFIGURASYONU
# ============================================================

# Lane bazli model listeleri
GENERALIST_MODELS_TO_TEST = [
    "qwen3-vl-2b",
    "qwen2.5-vl-3b",
    "qwen2-vl-2b",
    "phi3-vision",
    "smolvlm2-2.2b",
    "llama3-vision",  # gated Meta erisimin oldugu varsayimi
]

DOMAIN_ADAPTIVE_MODELS_TO_TEST = [
    "qwen2-vl-ocr-2b",
    "latxa-qwen3-vl-2b",
    "medgemma-4b",
]

SPECIALIST_MODELS_TO_TEST = [
    "got-ocr2",
    "nougat-base",
    "matcha-chartqa",
]

# Hangi lane'ler kosulsun? True/False yaparak secebilirsin.
RUN_LANES = {
    "generalist": True,
    "domain_adaptive": True,
    "specialist": True,
}

MODELS_TO_TEST = []
if RUN_LANES["generalist"]:
    MODELS_TO_TEST.extend(GENERALIST_MODELS_TO_TEST)
if RUN_LANES["domain_adaptive"]:
    MODELS_TO_TEST.extend(DOMAIN_ADAPTIVE_MODELS_TO_TEST)
if RUN_LANES["specialist"]:
    MODELS_TO_TEST.extend(SPECIALIST_MODELS_TO_TEST)

# Deney parametreleri
EXPERIMENT_NAME = "vqa_comparison_" + datetime.now().strftime("%Y%m%d_%H%M")
RESULTS_DIR = Path("results") / EXPERIMENT_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Deney: {EXPERIMENT_NAME}")
print(f"Sonuclar: {RESULTS_DIR}")
print(
    f"Generalist: {len(GENERALIST_MODELS_TO_TEST)} | "
    f"Domain-Adaptive: {len(DOMAIN_ADAPTIVE_MODELS_TO_TEST)} | "
    f"Specialist: {len(SPECIALIST_MODELS_TO_TEST)}"
)
print(f"Toplam aktif model ({len(MODELS_TO_TEST)}):")

for m in MODELS_TO_TEST:
    info = MODEL_REGISTRY.get(m)
    if info:
        print(f"   - {info.display_name} ({info.category.value}, {info.params})")

# Kullanilacak model tablosu
print("\n" + "=" * 60)
print_model_table()

---
## 4️⃣ Ana Deney Döngüsü — Model Karşılaştırması

Her modeli sırayla yükler → tüm örnekler üzerinde çıkarım yapar → GPU temizler → sonraki modele geçer.

In [8]:
def cleanup_gpu():
    """GPU belleğini temizle."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def run_vqa_inference(model, model_name, dataset, save_dir):
    """
    Bir modeli tüm veri seti üzerinde çalıştır.
    Her örnek için: image + question → model output
    """
    predictions = []
    references = []
    errors = 0
    total_time = 0

    for i in tqdm(range(len(dataset)), desc=f"{model_name}"):
        sample = dataset[i]
        img = sample["image"]
        question = sample["question"]
        answer = sample["answer"]

        # Görüntüyü geçici dosyaya kaydet (bazı modeller dosya yolu istiyor)
        tmp_path = "/tmp/eval_img.png"
        img.save(tmp_path)

        try:
            start = time.time()
            output = model.answer_question(tmp_path, question)
            elapsed = time.time() - start
            total_time += elapsed

            pred_text = output.text.strip()
            predictions.append(pred_text)
            references.append(answer)

        except Exception as e:
            logger.warning(f"Sample {i} hata: {e}")
            predictions.append("")
            references.append(answer)
            errors += 1

    # Geçici dosyayı temizle
    if os.path.exists(tmp_path):
        os.remove(tmp_path)

    # Sonuçları kaydet
    raw_results = {
        "model": model_name,
        "num_samples": len(dataset),
        "errors": errors,
        "total_time_sec": round(total_time, 2),
        "avg_time_per_sample": round(total_time / max(len(dataset), 1), 2),
        "predictions": predictions,
        "references": references,
    }

    with open(save_dir / f"{model_name}_raw.json", "w") as f:
        json.dump(raw_results, f, indent=2, ensure_ascii=False)

    return predictions, references, total_time, errors


print("✅ Inference fonksiyonları hazır.")

In [9]:
import builtins
from PIL import Image
import PIL.Image

# Image'ı tüm Python ortamına (builtins) enjekte et
builtins.Image = Image
builtins.PIL = PIL

print(f"✅ Image kütüphanesi sisteme (builtins) enjekte edildi: {PIL.Image.__version__}")
print("🚀 Artık tüm modeller 'Image' nesnesine her yerden erişebilir.")

In [10]:
%pip install -q transformers torch

In [11]:
# ============================================================
# 🚀 ANA DENEY DÖNGÜSÜ
# ============================================================

all_results = {}  # {model_name: {metric: value}}
all_predictions = {}  # {model_name: [pred1, pred2, ...]}
all_references = None  # Hep aynı

for model_name in MODELS_TO_TEST:
    print(f"\n{'='*60}")
    print(f"🤖 Model: {model_name}")
    print(f"{'='*60}")

    try:
        # 1) Model yükle
        print(f"   ⏳ Model indiriliyor ve yükleniyor...")
        model = load_model(model_name)
        model.load()
        print(f"   ✅ Model hazır!")

        # 2) Inference çalıştır
        print(f"   ⏳ {len(dataset)} örnek üzerinde çıkarım yapılıyor...")
        preds, refs, total_time, errors = run_vqa_inference(
            model, model_name, dataset, RESULTS_DIR
        )
        all_predictions[model_name] = preds
        all_references = refs  # Hep aynı, son set yeterli

        # 3) Metrikleri hesapla
        print(f"   📊 Metrikler hesaplanıyor...")

        # VQA Accuracy
        vqa_eval = VQAAccuracyEvaluator()
        vqa_scores = vqa_eval.compute(preds, refs)

        # BLEU
        bleu_eval = BLEUEvaluator()
        bleu_scores = bleu_eval.compute(preds, refs)

        # ROUGE
        rouge_eval = ROUGEEvaluator()
        rouge_scores = rouge_eval.compute(preds, refs)

        # Hallucination
        halluc_eval = FactCheXckerEvaluator()
        halluc_scores = halluc_eval.compute(preds, refs)

        # Sonuçları birleştir
        model_scores = {
            **vqa_scores,
            **bleu_scores,
            **rouge_scores,
            "hallucination_rate": halluc_scores.get("hallucination_rate", 0),
            "total_time_sec": round(total_time, 2),
            "avg_time_per_sample": round(total_time / max(len(dataset), 1), 2),
            "errors": errors,
        }
        all_results[model_name] = model_scores

        print(f"   ✅ Sonuçlar:")
        for metric, value in model_scores.items():
            if isinstance(value, float):
                print(f"      {metric}: {value:.4f}")
            else:
                print(f"      {metric}: {value}")

    except Exception as e:
        print(f"   ❌ HATA: {e}")
        all_results[model_name] = {"error": str(e)}

    finally:
        # 4) GPU belleğini temizle
        try:
            del model
        except:
            pass
        cleanup_gpu()
        print(f"   🧹 GPU temizlendi.")

print("\n" + "="*60)
print("🎉 Tüm modeller tamamlandı!")
print("="*60)

---
## 5️⃣ Sonuç Tablosu ve Analiz

In [12]:
# ============================================================
# 📊 KARŞILAŞTIRMA TABLOSU
# ============================================================

# Hatalı modelleri filtrele
valid_results = {k: v for k, v in all_results.items() if "error" not in v}

if valid_results:
    df = pd.DataFrame(valid_results).T

    # Kategori bilgisi ekle
    df.insert(0, "category", df.index.map(
        lambda m: MODEL_REGISTRY[m].category.value if m in MODEL_REGISTRY else "unknown"
    ))
    df.insert(1, "params", df.index.map(
        lambda m: MODEL_REGISTRY[m].params if m in MODEL_REGISTRY else "?"
    ))

    # Güzel format
    display_cols = ["category", "params", "accuracy", "bleu", "rouge_l",
                    "hallucination_rate", "avg_time_per_sample", "errors"]
    display_cols = [c for c in display_cols if c in df.columns]

    print("\n" + "="*80)
    print("📊 MODEL KARŞILAŞTIRMA SONUÇLARI")
    print("="*80)
    display(df[display_cols].round(4).sort_values("accuracy", ascending=False))

    # En iyi model
    if "accuracy" in df.columns:
        best_model = df["accuracy"].idxmax()
        print(f"\n🏆 En yüksek VQA Accuracy: {best_model} ({df.loc[best_model, 'accuracy']:.4f})")

else:
    print("❌ Hiçbir model başarıyla çalışmadı.")

# Hatalı modelleri göster
failed = {k: v for k, v in all_results.items() if "error" in v}
if failed:
    print("\n⚠️ Başarısız modeller:")
    for m, info in failed.items():
        print(f"   • {m}: {info['error'][:100]}")

In [13]:
# ============================================================
# 💾 SONUÇLARI KAYDET
# ============================================================

final_output = {
    "experiment": EXPERIMENT_NAME,
    "timestamp": datetime.now().isoformat(),
    "dataset": "VQA-RAD (HuggingFace)",
    "num_samples": len(dataset),
    "models_tested": MODELS_TO_TEST,
    "results": all_results,
}

output_path = RESULTS_DIR / "experiment_results.json"
with open(output_path, "w") as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print(f"✅ Sonuçlar kaydedildi: {output_path}")

# CSV olarak da kaydet (tablo için)
if valid_results:
    csv_path = RESULTS_DIR / "results_table.csv"
    df.to_csv(csv_path)
    print(f"✅ Tablo kaydedildi: {csv_path}")

---
## 6️⃣ Prior Bias Testi (Siyah Görüntü Deneyi)

**Hipotez:** Modeller görüntüye bakmadan, eğitim verisindeki ezber kalıplarla cevap verebilir.

**Yöntem:** Tamamen siyah (boş) bir görüntü verip "Is there any pleural effusion?" sorusu sorarak modelin yine de klinik cevap verip vermediğini test ediyoruz.

In [14]:
# ============================================================
# 🧪 PRIOR BIAS TESTI
# ============================================================

# Siyah görüntü oluştur
black_img = Image.new("RGB", (224, 224), color=(0, 0, 0))
black_img_path = "/tmp/black_test.png"
black_img.save(black_img_path)

bias_questions = [
    "Is there any pleural effusion?",
    "Is the heart size normal?",
    "Are the lungs clear?",
    "Generate a radiology report for this chest X-ray.",
]

# Hızlı lane modellerinden 2 tanesi ile bias testi
BIAS_MODELS = ["qwen2-vl-2b", "smolvlm2-2.2b"]

bias_results = {}

for model_name in BIAS_MODELS:
    print(f"\n{'='*50}")
    print(f"🧪 Bias Testi: {model_name}")
    print(f"{'='*50}")

    try:
        model = load_model(model_name)
        model.load()

        model_answers = []
        for q in bias_questions:
            try:
                output = model.answer_question(black_img_path, q)
                ans = output.text.strip()
                model_answers.append(ans)
                print(f"\n   ❓ Soru: {q}")
                print(f"   🤖 Cevap: {ans[:200]}")
            except Exception as e:
                model_answers.append(f"ERROR: {e}")
                print(f"   ❌ Hata: {e}")

        bias_results[model_name] = model_answers

        del model
        cleanup_gpu()

    except Exception as e:
        print(f"   ❌ Model yüklenemedi: {e}")
        bias_results[model_name] = [f"LOAD_ERROR: {e}"]

# Bias sonuçlarını kaydet
with open(RESULTS_DIR / "bias_test_results.json", "w") as f:
    json.dump(bias_results, f, indent=2, ensure_ascii=False)

print("\n✅ Bias testi tamamlandı ve kaydedildi.")

os.remove(black_img_path)

---
## 7️⃣ Prompt Ablasyon Çalışması

Aynı model ve aynı görüntü üzerinde farklı prompt stratejilerinin etkisini ölçüyoruz:
- **Baseline**: Minimal, doğrudan soru
- **Detailed**: Uzman rolü ve format yönergeli
- **Chain-of-Thought (CoT)**: Adım adım düşünme yönlendirmeli

In [15]:
# ============================================================
# 📝 PROMPT ABLASYONU
# ============================================================

prompt_manager = PromptManager()
prompt_variations = prompt_manager.get_all_prompt_variations("vqa")

ABLATION_MODEL = "qwen2-vl-2b"  # En hızlı model ile ablasyon
ABLATION_SAMPLES = 20  # İlk 20 örnek yeterli

print(f"🤖 Ablasyon modeli: {ABLATION_MODEL}")
print(f"📝 Prompt varyasyonları: {[p.name for p in prompt_variations]}")
print(f"📊 Örnek sayısı: {ABLATION_SAMPLES}")

ablation_results = {}

try:
    model = load_model(ABLATION_MODEL)
    model.load()

    for prompt_template in prompt_variations:
        prompt_name = prompt_template.name
        print(f"\n--- Prompt: {prompt_name} ---")

        preds = []
        refs = []

        for i in tqdm(range(min(ABLATION_SAMPLES, len(dataset)))):
            sample = dataset[i]
            img = sample["image"]
            question = sample["question"]

            # Prompt'u formatla
            formatted = prompt_template.format(question=question)
            full_prompt = formatted["user"]

            tmp_path = "/tmp/ablation_img.png"
            img.save(tmp_path)

            try:
                output = model.answer_question(tmp_path, full_prompt)
                preds.append(output.text.strip())
            except Exception as e:
                preds.append("")
            refs.append(sample["answer"])

        # Metrikleri hesapla
        vqa_scores = VQAAccuracyEvaluator().compute(preds, refs)
        bleu_scores = BLEUEvaluator().compute(preds, refs)

        ablation_results[prompt_name] = {
            "accuracy": vqa_scores.get("accuracy", 0),
            "bleu": bleu_scores.get("bleu", 0),
            "sample_predictions": preds[:5],
        }

        print(f"   Accuracy: {vqa_scores.get('accuracy', 0):.4f}")
        print(f"   BLEU: {bleu_scores.get('bleu', 0):.4f}")

    del model
    cleanup_gpu()

except Exception as e:
    print(f"❌ Ablasyon hatası: {e}")

# Ablasyon sonuçlarını kaydet
with open(RESULTS_DIR / "prompt_ablation_results.json", "w") as f:
    json.dump(ablation_results, f, indent=2, ensure_ascii=False)

if os.path.exists("/tmp/ablation_img.png"):
    os.remove("/tmp/ablation_img.png")

# Ablasyon tablosu
if ablation_results:
    ablation_df = pd.DataFrame(ablation_results).T[["accuracy", "bleu"]]
    print("\n" + "="*50)
    print("📝 PROMPT ABLASYON SONUÇLARI")
    print("="*50)
    display(ablation_df.round(4))

---
## 8️⃣ Kategori Bazlı Analiz

Generalist vs Domain-Adaptive vs Specialist karşılaştırması.

In [16]:
# ============================================================
# 📊 KATEGORİ ANALİZİ
# ============================================================

if valid_results:
    category_data = {}
    for model_name, scores in valid_results.items():
        info = MODEL_REGISTRY.get(model_name)
        if info:
            cat = info.category.value
            if cat not in category_data:
                category_data[cat] = []
            category_data[cat].append({
                "model": model_name,
                "accuracy": scores.get("accuracy", 0),
                "bleu": scores.get("bleu", 0),
                "rouge_l": scores.get("rouge_l", 0),
                "hallucination_rate": scores.get("hallucination_rate", 0),
            })

    print("\n" + "="*60)
    print("📊 KATEGORİ BAZLI KARŞILAŞTIRMA")
    print("="*60)

    for cat, models in category_data.items():
        avg_acc = np.mean([m["accuracy"] for m in models])
        avg_bleu = np.mean([m["bleu"] for m in models])
        avg_halluc = np.mean([m["hallucination_rate"] for m in models])
        print(f"\n🏷️  {cat.upper()} (n={len(models)})")
        print(f"   Avg Accuracy:          {avg_acc:.4f}")
        print(f"   Avg BLEU:              {avg_bleu:.4f}")
        print(f"   Avg Hallucination:     {avg_halluc:.4f}")
        for m in models:
            print(f"      • {m['model']}: acc={m['accuracy']:.4f}")

else:
    print("❌ Analiz için yeterli sonuç yok.")

---
## 📋 Deney Özeti

Tüm sonuçlar `results/` klasörü altında kaydedilmiştir:
- `experiment_results.json` — Ana sonuç dosyası
- `results_table.csv` — Tablo formatında sonuçlar
- `bias_test_results.json` — Prior bias testi sonuçları
- `prompt_ablation_results.json` — Prompt ablasyon sonuçları
- `{model_name}_raw.json` — Her modelin ham çıktıları

Bu dosyaları doğrudan makale tabloları ve grafikleri için kullanabilirsiniz.

In [17]:
# Dosyaları listele
print("📁 Kaydedilen dosyalar:")
for f in sorted(RESULTS_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name} ({size_kb:.1f} KB)")